In [ ]:
import sys
project_root = "/usr/local/google/home/tzengy/ml-auto-solutions-3"
if project_root not in sys.path:
    sys.path.append(project_root)

from google.cloud.monitoring_v3 import types
from dags.tpu_observability.utils.time_util import TimeUtil
from dags.tpu_observability.utils.gcp_util import list_time_series
from dataclasses import dataclass

@dataclass
class NodePoolInfo:
    project_id: str
    cluster_name: str
    location: str
    region: str
    node_locations: str
    zone: str
    reservation: str
    num_nodes: int

def test_fetch_replicated_job_availability(node_pool_info: NodePoolInfo) -> None:
    """
    測試用的函式：尋找 kubernetes.io/internal/crd/replicated_job/availability
    並固定查詢 2026-08-24 00:00:00 UTC 到 2026-08-24 02:00:00 UTC 的資料，然後印出結果。
    """
    # 設定固定的 UTC 時間範圍
    start_time = TimeUtil.from_iso_string("2026-08-24T00:00:00Z")
    end_time = TimeUtil.from_iso_string("2026-08-24T02:00:00Z")

    metric_name = "kubernetes.io/internal/crd/replicated_job/availability"

    # 你可以依照需求把 cluster_name 加回 filter 裡
    filter_string = [
        f'metric.type = "{metric_name}"',
        f'resource.labels.cluster_name = "{node_pool_info.cluster_name}"',
    ]

    print(f"Project ID: {node_pool_info.project_id}")
    print(f"Start time: {start_time.to_iso_string()}")
    print(f"End time: {end_time.to_iso_string()}")
    print(f"Filter: {' AND '.join(filter_string)}")
    print("-" * 50)

    try:
        # 呼叫已經存在的 list_time_series 函式
        time_series_data = list_time_series(
            project_id=node_pool_info.project_id,
            filter_str=" AND ".join(filter_string),
            start_time=start_time,
            end_time=end_time,
            view=types.ListTimeSeriesRequest.TimeSeriesView.FULL,
        )

        if not time_series_data:
            print("No data found for this metric in the specified time range.")
            return

        print(f"Found {len(time_series_data)} time series matching the criteria.")

        # 逐一印出每個 time series
        for ts_index, ts in enumerate(time_series_data):
            print(f"\n[Time Series {ts_index + 1}]")
            print(f"Metric labels: {ts.metric.labels}")
            print(f"Resource labels: {ts.resource.labels}")
            print(f"Total data points: {len(ts.points)}")

            # 為了避免資料量太大，這裡只印出前 10 筆資料點
            print("Data points (up to first 10):")
            for point in ts.points[:10]:
                point_time = point.interval.end_time.strftime('%Y-%m-%dT%H:%M:%SZ')

                # 依據資料型態取值
                if point.value.HasField("int64_value"):
                    val = point.value.int64_value
                elif point.value.HasField("double_value"):
                    val = point.value.double_value
                else:
                    val = point.value

                print(f"  - Time: {point_time} | Value: {val}")

    except Exception as e:
        print(f"Error fetching time series: {e}")

# Create the NodePoolInfo instance with user's data
my_node_pool = NodePoolInfo(
    project_id="cienet-cmcs",
    cluster_name="yuna-automation",
    location="us-central1",
    region="us-central1",
    node_locations="us-central1-b",
    zone="us-central1-b",
    reservation="",
    num_nodes=4
)

# Call the function
test_fetch_replicated_job_availability(my_node_pool)